In [4]:
import numpy as np
import pandas as pd
from sklearn.model_selection import KFold
from scipy.optimize import minimize_scalar, minimize
from statsmodels.nonparametric.kernel_regression import KernelReg

# ==========================================
# КОД МЕТОДОВ (БЕЗ ИЗМЕНЕНИЙ)
# ==========================================

def kernel_regression_silverman(x_train, y_train):
    """Ядерная регрессия с правилом Сильвермана."""
    x_train = np.asarray(x_train, dtype=np.float64).reshape(-1, 1)
    y_train = np.asarray(y_train, dtype=np.float64)
    
    n = len(x_train)
    h = 1.06 * np.std(x_train) * n**(-1/5)
    
    model = KernelReg(y_train, x_train, var_type='c', bw=[h])
    return model, h

def kernel_regression_cv(x_train, y_train):
    """Ядерная регрессия с кросс-валидацией."""
    x_train = np.asarray(x_train, dtype=np.float64).reshape(-1, 1)
    y_train = np.asarray(y_train, dtype=np.float64)
    
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    def cv_mse(log_h):
        h = np.exp(log_h)
        
        total_mse = 0.0
        for train_idx, val_idx in kf.split(x_train):
            kr = KernelReg(y_train[train_idx], x_train[train_idx], var_type='c', bw=[h])
            pred = kr.fit(x_train[val_idx])[0]
            total_mse += np.mean((pred - y_train[val_idx])**2)
        
        return total_mse / kf.n_splits
    
    n = len(x_train)
    h_silverman = 1.06 * np.std(x_train) * n**(-1/5)
    log_h_center = np.log(h_silverman)
    bounds = (log_h_center - 5, log_h_center + 5) 
    res = minimize_scalar(cv_mse, bounds=bounds, method='bounded')
    
    if res.success:
        best_h = np.exp(res.x)
    else:
        n = len(x_train)
        best_h = 1.06 * np.std(x_train) * n**(-1/5)
    
    model = KernelReg(y_train, x_train, var_type='c', bw=[best_h])
    return model, best_h

def kernel_regression_loocv(x_train, y_train):
    """Ядерная регрессия с методом скользящего контроля (LOOCV)."""
    x_train = np.asarray(x_train, dtype=np.float64).reshape(-1, 1)
    y_train = np.asarray(y_train, dtype=np.float64)
    
    n = len(x_train)
    
    def loocv_mse(log_h):
        h = np.exp(log_h)
        total_error = 0.0
        
        for i in range(n):
            x_oob = x_train[i:i+1]
            y_oob = y_train[i]
            
            x_train_oob = np.delete(x_train, i, axis=0)
            y_train_oob = np.delete(y_train, i)
            
            kr = KernelReg(y_train_oob, x_train_oob, var_type='c', bw=[h])
            pred = kr.fit(x_oob)[0]
            total_error += (pred - y_oob) ** 2
        
        return total_error / n
    
    n = len(x_train)
    h_silverman = 1.06 * np.std(x_train) * n**(-1/5)
    log_h_center = np.log(h_silverman)
    bounds = (log_h_center - 5, log_h_center + 5) 
    res = minimize_scalar(loocv_mse, bounds=bounds, method='bounded')
    
    if res.success:
        best_h = float(np.exp(res.x))
    else:
        best_h = float(1.06 * np.std(x_train) * n**(-1/5))
    
    model = KernelReg(y_train, x_train, var_type='c', bw=[best_h])
    return model, best_h

def create_predictor(model):
    """Создаёт предиктор для метрик."""
    def predictor(x):
        x_arr = np.asarray(x, dtype=np.float64).reshape(-1, 1)
        return model.fit(x_arr)[0]
    return predictor

def mixed_norm_polyfit(x, y, c1, c2, degree):
    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)
    
    if x.shape[0] == 0 or y.shape[0] == 0:
        return np.zeros(degree + 1), np.array([])
    
    if x.shape[0] != y.shape[0]:
        raise ValueError("Длины x и y должны совпадать")
    
    if c1 < 0 or c2 < 0:
        raise ValueError("Коэффициенты c1 и c2 должны быть неотрицательными")
    
    if c2 == 0:
        c2 = 1e-15
    
    X = np.vander(x, N=degree + 1, increasing=True)
    
    try:
        coef0, _, _, _ = np.linalg.lstsq(X, y, rcond=None)
    except np.linalg.LinAlgError:
        coef0 = np.zeros(degree + 1)
    
    if c1 == 0 and c2 == 0:
        return coef0, X @ coef0
    
    def loss(coef):
        residuals = X @ coef - y
        if c2 > 0 and np.any(np.abs(residuals) > 1e10):
            return np.inf
        l1_term = np.sum(np.abs(residuals))
        l2_term = np.sum(residuals ** 2) if c2 > 0 else 0.0
        return c1 * l1_term + c2 * l2_term
    
    result = minimize(loss, coef0, method='Powell', options={'maxiter': 10000, 'disp': False})
    
    if not result.success:
        result = minimize(loss, coef0, method='Nelder-Mead', options={'maxiter': 10000, 'disp': False})
    
    coefficients = result.x
    y_pred = X @ coefficients
    
    return coefficients, y_pred

def full_grid_bootstrap_selection(x, y, degrees=(1, 2, 3, 4, 5, 6),
                                  B=200, random_state=42):
    np.random.seed(random_state)
    
    ratios = [(1, 0), (0, 1), (1, 1), (3, 4), (4, 3), 
              (1, 2), (2, 1), (1, 4), (4, 1)]
    grid = [(a / (a + b), b / (a + b)) for a, b in ratios]
    
    n = len(x)
    results = {}
    
    total_combinations = len(degrees) * len(grid)
    current = 0
    
    for degree in degrees:
        for c1, c2 in grid:
            current += 1
            
            err_train_list = []
            err_boot_list = []
            gamma_list = []
            
            for b in range(B):
                indices = np.random.choice(n, size=n, replace=True)
                x_boot = x[indices]
                y_boot = y[indices]
                out_of_bag = np.setdiff1d(np.arange(n), indices)
                
                coef, y_pred_boot = mixed_norm_polyfit(x_boot, y_boot, c1, c2, degree)
                
                err_train = np.mean((y_boot - y_pred_boot) ** 2)
                
                if len(out_of_bag) > 0:
                    X_oob = np.vander(x[out_of_bag], N=degree + 1, increasing=True)
                    y_pred_oob = X_oob @ coef
                    err_boot = np.mean((y[out_of_bag] - y_pred_oob) ** 2)
                else:
                    err_boot = err_train
                
                X_full = np.vander(x, N=degree + 1, increasing=True)
                y_pred_full = X_full @ coef
                
                y_pred_matrix = np.tile(y_pred_full, (n, 1))
                y_true_matrix = np.tile(y.reshape(-1, 1), (1, n))
                gamma = np.mean((y_true_matrix - y_pred_matrix.T) ** 2)
                
                err_train_list.append(err_train)
                err_boot_list.append(err_boot)
                gamma_list.append(gamma)
            
            err_train_avg = np.mean(err_train_list)
            err_boot_avg = np.mean(err_boot_list)
            gamma_avg = np.mean(gamma_list)
            
            R = (err_boot_avg - err_train_avg) / (gamma_avg - err_train_avg + 1e-10)
            R = np.clip(R, 0, 1)
            
            w = 0.632 / (1 - 0.368 * R)
            w = np.clip(w, 0.632, 1.0)
            
            err_632plus = (1 - w) * err_train_avg + w * err_boot_avg
            
            results[(degree, c1, c2)] = {
                'err_632plus': err_632plus,
                'err_train': err_train_avg,
                'err_boot': err_boot_avg,
                'gamma': gamma_avg,
                'R': R,
                'w': w
            }
    
    best_key = min(results.keys(), key=lambda k: results[k]['err_632plus'])
    best_degree, best_c1, best_c2 = best_key
    
    return best_degree, best_c1, best_c2

# ==========================================
# ОЦЕНКА МЕТОДОВ С ПАРАМЕТРАМИ
# ==========================================

def calculate_metrics(y_true, y_pred):
    """Расчет метрик RMSE, MAE, MaxErr."""
    errors = y_true - y_pred
    rmse = np.sqrt(np.mean(errors ** 2))
    mae = np.mean(np.abs(errors))
    max_err = np.max(np.abs(errors))
    return rmse, mae, max_err

def evaluate_method_5fold(X, y, method_func, method_name):
    """Оценка метода с помощью 5-fold кросс-валидации с сохранением параметров."""
    kf = KFold(n_splits=5, shuffle=True, random_state=42)
    
    rmse_list, mae_list, max_err_list = [], [], []
    params_list = []
    
    for fold, (train_idx, test_idx) in enumerate(kf.split(X)):
        X_train, X_test = X[train_idx], X[test_idx]
        y_train, y_test = y[train_idx], y[test_idx]
        
        if method_name == "Entropy":
            degree, c1, c2 = full_grid_bootstrap_selection(X_train, y_train, B=200, random_state=42)
            coef, _ = mixed_norm_polyfit(X_train, y_train, c1, c2, degree)
            
            params = {
                'fold': fold + 1,
                'degree': degree,
                'c1': c1,
                'c2': c2,
                'coefficients': ', '.join([f'{c:.6f}' for c in coef])
            }
            
            def predictor(x):
                x_arr = np.asarray(x, dtype=np.float64)
                X_vander = np.vander(x_arr, N=degree + 1, increasing=True)
                return X_vander @ coef
        else:
            model, h = method_func(X_train, y_train)
            predictor = create_predictor(model)
            
            params = {
                'fold': fold + 1,
                'bandwidth_h': h
            }
        
        y_pred = predictor(X_test)
        rmse, mae, max_err = calculate_metrics(y_test, y_pred)
        
        rmse_list.append(rmse)
        mae_list.append(mae)
        max_err_list.append(max_err)
        params_list.append(params)
    
    metrics = {
        'Method': method_name,
        'RMSE': np.mean(rmse_list),
        'MAE': np.mean(mae_list),
        'MaxErr': np.mean(max_err_list)
    }
    
    return metrics, params_list

# ==========================================
# ЗАПУСК ОЦЕНКИ
# ==========================================

if __name__ == "__main__":
    # Загрузка данных
    df = pd.read_csv('faithful.csv')
    X = df['waiting'].values
    y = df['eruptions'].values
    
    # === ДОБАВЛЕНО: Нормализация ===
    X_mean, X_std = X.mean(), X.std()
    y_mean, y_std = y.mean(), y.std()
    
    X_normalized = (X - X_mean) / X_std
    y_normalized = (y - y_mean) / y_std
    # ================================
    
    # Определение методов для оценки
    methods = [
        (kernel_regression_silverman, "Silverman"),
        (kernel_regression_cv, "Kernel_5Fold_CV"),
        (kernel_regression_loocv, "Kernel_LOOCV"),
        (None, "Entropy")
    ]
    
    all_metrics = []
    all_params = []
    
    print("Оценка методов (5-fold CV)...")
    for method_func, method_name in methods:
        print(f"  - {method_name}...", end=' ', flush=True)
        # === ИЗМЕНЕНО: используем нормализованные данные ===
        metrics, params = evaluate_method_5fold(X_normalized, y_normalized, method_func, method_name)
        # =====================================================
        all_metrics.append(metrics)
        
        for p in params:
            p['Method'] = method_name
            all_params.append(p)
        
        print(f"RMSE={metrics['RMSE']:.4f}")
    
    results_df = pd.DataFrame(all_metrics)
    params_df = pd.DataFrame(all_params)
    
    print("\n" + "="*60)
    print("МЕТРИКИ КАЧЕСТВА (нормализованные данные):")
    print("="*60)
    print(results_df.to_string(index=False))
    
    print("\n" + "="*60)
    print("ПОДОБРАННЫЕ ПАРАМЕТРЫ:")
    print("="*60)
    print(params_df.to_string(index=False))
    
    results_df.to_csv('regression_metrics_results_normalized.csv', index=False)
    params_df.to_csv('regression_parameters_results_normalized.csv', index=False)
    
    print("\n✓ Результаты сохранены в 'regression_metrics_results_normalized.csv'")
    print("✓ Параметры сохранены в 'regression_parameters_results_normalized.csv'")

Оценка методов (5-fold CV)...
  - Silverman... RMSE=0.3386
  - Kernel_5Fold_CV... RMSE=0.3318
  - Kernel_LOOCV... 

C:\Users\Dima\AppData\Local\Temp\ipykernel_19948\4251076778.py:86: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  best_h = float(np.exp(res.x))


RMSE=0.3311
  - Entropy... RMSE=0.3465

МЕТРИКИ КАЧЕСТВА (нормализованные данные):
         Method     RMSE      MAE   MaxErr
      Silverman 0.338562 0.268040 0.913241
Kernel_5Fold_CV 0.331764 0.260397 0.954814
   Kernel_LOOCV 0.331069 0.260099 0.955194
        Entropy 0.346494 0.268775 0.995731

ПОДОБРАННЫЕ ПАРАМЕТРЫ:
 fold  bandwidth_h          Method  degree       c1       c2                                                            coefficients
    1     0.370668       Silverman     NaN      NaN      NaN                                                                     NaN
    2     0.367729       Silverman     NaN      NaN      NaN                                                                     NaN
    3     0.355986       Silverman     NaN      NaN      NaN                                                                     NaN
    4     0.358892       Silverman     NaN      NaN      NaN                                                                     NaN
    5     0.3